# Spotify Hit & Mood Analysis

An interactive companion to the dashboard. This notebook explores the supplied track dataset, trains the same hit classifiers used by the app, and visualizes the five audio-feature mood clusters. The model-training cell includes cross-validation and may take a few minutes.

Run from the project root (or its subfolder) after installing `requirements.txt`, then execute the cells from top to bottom.

In [ ]:
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

DATA_CANDIDATES = [parent / "data" / "spotify_tracks.csv" for parent in (Path.cwd(), *Path.cwd().parents)]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/spotify_tracks.csv from this notebook's working directory.")

TARGET_THRESHOLD = 60
NUMERIC = [
    "danceability", "energy", "key", "loudness", "mode", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
    "time_signature", "duration_ms",
]
AUDIO = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo",
]
print(f"Dataset: {DATA_PATH}")

## Load and check the data

Numeric columns are coerced consistently with `app/ml.py`. Rows without a popularity score cannot be used for the hit target and are removed.

In [ ]:
df = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore")
for column in NUMERIC + ["popularity"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["track_genre"] = df["track_genre"].fillna("unknown").astype(str)
df["artists"] = df["artists"].fillna("Unknown artist")
df["track_name"] = df["track_name"].fillna("Untitled")
df = df.dropna(subset=["popularity"]).copy()

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
display(missing_values[missing_values > 0].to_frame("missing values"))
display(df[NUMERIC + ["popularity"]].describe().T)

## Popularity and genre overview

A hit is defined as a track with Spotify popularity of at least 60, matching the API and model target.

In [ ]:
df["is_hit"] = (df["popularity"] >= TARGET_THRESHOLD).astype(int)
print(f"Hit rate: {df['is_hit'].mean():.1%}")

top_genres = (
    df.groupby("track_genre")
    .agg(tracks=("track_id", "size"), mean_popularity=("popularity", "mean"), hit_rate=("is_hit", "mean"))
    .sort_values("mean_popularity", ascending=False)
    .head(12)
)
display(top_genres.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df["popularity"], bins=20, color="#1DB954", edgecolor="white")
axes[0].axvline(TARGET_THRESHOLD, color="#191414", linestyle="--", label="Hit threshold (60)")
axes[0].set(title="Track popularity", xlabel="Popularity", ylabel="Tracks")
axes[0].legend()
top_genres.sort_values("mean_popularity")["mean_popularity"].plot.barh(ax=axes[1], color="#1DB954")
axes[1].set(title="Top genres by mean popularity", xlabel="Mean popularity", ylabel="")
plt.tight_layout()
plt.show()

## Train and evaluate hit classifiers

To keep training practical, a reproducible sample of up to 24,000 rows is used. The split is stratified (78% train / 22% test). Boosting uses a randomized five-trial, three-fold F1 search, as in `app/ml.py`.

In [ ]:
work = df.sample(n=min(24_000, len(df)), random_state=42)
X = work[NUMERIC + ["track_genre"]].copy()
y = (work["popularity"] >= TARGET_THRESHOLD).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.22, random_state=42, stratify=y
)

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC),
    ("genre", OneHotEncoder(handle_unknown="ignore"), ["track_genre"]),
])
models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=1200, class_weight="balanced")),
    ]),
    "KNN": Pipeline([
        ("prep", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=13, weights="distance", n_jobs=-1)),
    ]),
}
for model in models.values():
    model.fit(X_train, y_train)

boost_preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median"))]), NUMERIC),
    ("genre", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), ["track_genre"]),
])
boost_pipeline = Pipeline([
    ("prep", boost_preprocessor),
    ("model", HistGradientBoostingClassifier(random_state=42)),
])
search = RandomizedSearchCV(
    boost_pipeline,
    {
        "model__learning_rate": [0.04, 0.07, 0.1],
        "model__max_iter": [100, 160, 220],
        "model__max_leaf_nodes": [15, 31],
        "model__l2_regularization": [0, 0.5, 1.0],
    },
    n_iter=5, cv=3, scoring="f1", random_state=42, n_jobs=-1, refit=True,
)
search.fit(X_train, y_train)
models["Tuned Boosting"] = search.best_estimator_
print("Best boosting parameters:", search.best_params_)

In [ ]:
metric_rows = {}
for name, model in models.items():
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    metric_rows[name] = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
    }
metrics = pd.DataFrame.from_dict(metric_rows, orient="index").sort_values("f1", ascending=False)
display(metrics.round(4))

## Discover audio-feature moods

K-Means groups standardized audio features into five clusters. PCA reduces those features to two dimensions for visualization only; the clusters are fit in the original standardized feature space.

In [ ]:
mood_data = df.dropna(subset=AUDIO)
mood_data = mood_data.sample(n=min(7_000, len(mood_data)), random_state=17).copy()
mood_scaler = StandardScaler()
Z = mood_scaler.fit_transform(mood_data[AUDIO])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
mood_data["cluster"] = kmeans.fit_predict(Z)
pca = PCA(n_components=2, random_state=42)
coordinates = pca.fit_transform(Z)
mood_data["pca_1"] = coordinates[:, 0]
mood_data["pca_2"] = coordinates[:, 1]

cluster_means = mood_data.groupby("cluster")[["energy", "valence", "danceability", "acousticness"]].mean()

def describe_mood(row):
    if row["energy"] >= 0.68 and row["valence"] >= 0.55:
        return "Euphoric"
    if row["energy"] >= 0.68:
        return "High Energy"
    if row["valence"] >= 0.58:
        return "Feel Good"
    if row["acousticness"] >= 0.55:
        return "Chill & Acoustic"
    return "Melancholic"

mood_names = {cluster: describe_mood(row) for cluster, row in cluster_means.iterrows()}
mood_data["mood"] = mood_data["cluster"].map(mood_names)
mood_summary = mood_data.groupby("mood").agg(
    tracks=("track_id", "size"),
    mean_energy=("energy", "mean"),
    mean_valence=("valence", "mean"),
    mean_danceability=("danceability", "mean"),
    mean_acousticness=("acousticness", "mean"),
).sort_values("tracks", ascending=False)
display(mood_summary.round(3))
print("PCA explained variance:", np.round(pca.explained_variance_ratio_, 4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for mood, points in mood_data.groupby("mood"):
    ax.scatter(points["pca_1"], points["pca_2"], s=10, alpha=0.55, label=mood)
ax.set(title="Spotify tracks projected into mood space", xlabel="PCA component 1", ylabel="PCA component 2")
ax.legend(title="Mood", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Example hit prediction

Use one held-out track to compare the three model probabilities and their mean consensus.

In [ ]:
sample_index = X_test.index[0]
sample_track = X_test.loc[[sample_index]]
sample_details = work.loc[sample_index, ["track_name", "artists", "track_genre", "popularity"]]
hit_probabilities = {
    name: float(model.predict_proba(sample_track)[0, 1])
    for name, model in models.items()
}
prediction_summary = pd.DataFrame.from_dict(
    hit_probabilities, orient="index", columns=["hit probability"]
)
prediction_summary.loc["Consensus mean"] = np.mean(list(hit_probabilities.values()))
display(sample_details.to_frame("track"))
display(prediction_summary.round(3))
print("Actual label:", "Hit" if y_test.loc[sample_index] else "Not a hit")